# Introduction

This notebook aims to understand and visualize static GTFS data reported from the TTC
Specifically we would like to visualize Route 29, a bus route that is known to have significantly high trip times

You can find the exact data source used in this project below: https://open.toronto.ca/dataset/merged-gtfs-ttc-routes-and-schedules/



# Data Modeling

Before we dig into the data itself, we will start off by understanding the inherent structure of the data (the data model). The GTFS specifciation is a global standard used to structure transit data reported by transit agencies. Specifically the GTFS specifies an array of CSV text files each representing different types of transit information (i.e. route/trip information, schedule information, geo-shape information etc.). Each CSV text file relate to each other through a relational schema.

A relational schema representing the GTFS data reported by the TTC can be found below:


![GTFS_STATIC_Relational_Model](/Users/devrajsolanki/Documents/TAL/GTFS_STATIC_Relational_Model.jpeg)

# Data Preprocessing

GTFS feeds can be quite messy at times, containing incomplete, invalid and inconsistent data.
Thankfully there is significant effort to validate the quality of GTFS feed data.
 
We will be using validation techniques suggested from Mobility Data (https://gtfs-validator.mobilitydata.org/rules.html) as well as common 
Data Preprocessing/Cleaning techniques.



### Data Loading

In [120]:
import pandas as pd

In [121]:
stops = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stops.txt")
routes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/routes.txt")
trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")
shapes = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/shapes.txt")
calendar = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar.txt")
agency = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/agency.txt")
calendar_dates = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/calendar_dates.txt")
route_types = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/route_types.txt")
feed_info = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/feed_info.txt")

/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_73779/2009019488.py:3: DtypeWarning: Columns (4,7) have mixed types. Specify dtype option on import or set low_memory=False.
  trips = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/trips.txt")
/var/folders/b2/y50nhjkn7554xcj0cffkg3m80000gn/T/ipykernel_73779/2009019488.py:4: DtypeWarning: Columns (5) have mixed types. Specify dtype option on import or set low_memory=False.
  stop_times = pd.read_csv("/Users/devrajsolanki/Documents/Complete_GTFS/stop_times.txt")


### Data Understanding and Cleaning 

The inclusion of a stop_times.txt file means this is a schedule-based feed (define specific arrival and departure times for every trip at every stop)

In [122]:
# lets look at the shape of each dataframe to understand the size of the data

print("Shape of stops dataframe:", stops.shape)
print("Shape of routes dataframe:", routes.shape)
print("Shape of trips dataframe:", trips.shape)
print("Shape of stop_times dataframe:", stop_times.shape)
print("Shape of shapes dataframe:", shapes.shape)
print("Shape of calendar dataframe:", calendar.shape)
print("Shape of agency dataframe:", agency.shape)
print("Shape of calendar_dates dataframe:", calendar_dates.shape)
print("Shape of route_types dataframe:", route_types.shape)
print("Shape of feed_info dataframe:", feed_info.shape)

# It seems that the number of records for each file matches logic; there should be a lot of stop_times records since each trip has multiple 
# stops and there should be fewer routes than trips, etc. This is a good sign that the feed is complete.

Shape of stops dataframe: (9423, 13)
Shape of routes dataframe: (230, 9)
Shape of trips dataframe: (135412, 9)
Shape of stop_times dataframe: (4283994, 10)
Shape of shapes dataframe: (1020671, 5)
Shape of calendar dataframe: (8, 10)
Shape of agency dataframe: (1, 9)
Shape of calendar_dates dataframe: (9, 3)
Shape of route_types dataframe: (679, 5)
Shape of feed_info dataframe: (1, 6)


#### Stops.txt

In [123]:
# lets look at the data types of each column in each dataframe to identify if the feed is clean/recorded properly

print("Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_desc              float64
stop_lat               float64
stop_lon               float64
zone_id                float64
stop_url               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [124]:
# its weird that stop_desc is a float and not a string/object, so lets look at the unique values in the column
print("Unique values in stop_desc column of stops dataframe:")
print(stops["stop_desc"].unique())

# so it seems that the TTC Feed leaves out/excludes stop description, so lets remove the column from the dataframe
stops = stops.drop(columns=["stop_desc"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Unique values in stop_desc column of stops dataframe:
[nan]
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
zone_id                float64
stop_url               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [125]:
# everything checks out, lets now look at the distribution of values for each column as well
# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

# stop_id
print("Distribution of stop_id column in stops dataframe:")
print(stops["stop_id"].value_counts())
print("-" * 50)

# There are 9423 unique stop_ids which matches the number of records in the stop dataframe so there are no duplicate primary keys

# stop_code
print("Distribution of stop_code column in stops dataframe:")
print(stops["stop_code"].value_counts())
print("-" * 50)

# There also seems to be 9423 unique stop_codes per record

Distribution of stop_id column in stops dataframe:
stop_id
1        1
8449     1
8430     1
8431     1
8432     1
        ..
4742     1
4743     1
4745     1
4746     1
16803    1
Name: count, Length: 9423, dtype: int64
--------------------------------------------------
Distribution of stop_code column in stops dataframe:
stop_code
7978     1
5831     1
12194    1
990      1
2395     1
        ..
3959     1
8896     1
1876     1
7166     1
16803    1
Name: count, Length: 9423, dtype: int64
--------------------------------------------------


In [126]:
# stop_name
print("Distribution of stop_name column in stops dataframe:")
print(stops["stop_name"].value_counts())
print("-" * 50)

# filter stop_name for "Centennial College Bus Terminal"
centennial_stops = stops[stops["stop_name"] == "Centennial College Bus Terminal"]
print("Filtered stops for 'Centennial College Bus Terminal':")
print(centennial_stops)

# So it seems that some stops share the same name but have different stop_ids and stop_codes and even slightly different coordinates. 
# This makes sense since there can be multiple platforms at the same general location (Centennial college has 4 bus platforms).

Distribution of stop_name column in stops dataframe:
stop_name
Centennial College Bus Terminal                                 4
Humber College Bus Terminal                                     4
The Queensway at South Kingsway                                 4
Bathurst St at King St West                                     4
Bathurst St at Niagara St                                       4
                                                               ..
Millwood Rd at Southvale Dr                                     1
Belfield Rd at City View Dr                                     1
Faywood Blvd at Invermay Ave                                    1
Queens Quay West at Bathurst St East Side Billy Bishop Airpo    1
McCowan Rd at Highway 401                                       1
Name: count, Length: 7782, dtype: int64
--------------------------------------------------
Filtered stops for 'Centennial College Bus Terminal':
      stop_id  stop_code                        stop_name   stop_l

In [127]:
# stop_lat
print("Distribution of stop_lat column in stops dataframe:")
print("Range of stop_lat:", stops["stop_lat"].min(), "to", stops["stop_lat"].max())
print("-" * 50)

# stop_lon
print("Distribution of stop_lon column in stops dataframe:")
print("Range of stop_lon:", stops["stop_lon"].min(), "to", stops["stop_lon"].max())
print("-" * 50)

# Toronto lies around 43.6425, -79.387222 [wikipedia] so the stop coordinates seem to be within the expected range for Toronto.

Distribution of stop_lat column in stops dataframe:
Range of stop_lat: 43.59181 to 43.90975
--------------------------------------------------
Distribution of stop_lon column in stops dataframe:
Range of stop_lon: -79.649908 to -79.123111
--------------------------------------------------


In [128]:
# zone_id
print("Distribution of zone_id column in stops dataframe:")
print(stops["zone_id"].value_counts())
print("-" * 50)

# It seems that the zone_id column is not used in the feed since all values are NaN,

# stop_url
print("Distribution of stop_url column in stops dataframe:")
print(stops["stop_url"].value_counts())
print("-" * 50)

# It seems that the stop_url column is not used in the feed since all values are NaN

# remove the zone_id and stop_url columns since they are not used in the feed
stops = stops.drop(columns=["zone_id", "stop_url"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)


Distribution of zone_id column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Distribution of stop_url column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
parent_station         float64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [129]:
# location_type
print("Distribution of location_type column in stops dataframe:")
print(stops["location_type"].value_counts())
print("-" * 50)

# so it seems that the TTC reported all stops as location_type 0 (stops/platforms); this means that stations, entrances/exits are 
# not included in the feed data; just the points where the bus/subway etc. picks up/drops off passengers

# parent_station
print("Distribution of parent_station column in stops dataframe:")
print(stops["parent_station"].value_counts())
print("-" * 50)

# hence parent_station has all values are NaN since stations are not included in the feed data 

# drop parent_station column since it is not used in the feed
stops = stops.drop(columns=["parent_station"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)

Distribution of location_type column in stops dataframe:
location_type
0    9423
Name: count, dtype: int64
--------------------------------------------------
Distribution of parent_station column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
stop_timezone          float64
wheelchair_boarding      int64
level_id               float64
dtype: object
--------------------------------------------------


In [130]:
# stop_timezone
print("Range of stop_timezone column in stops dataframe:")
print(stops["stop_timezone"].min(), stops["stop_timezone"].max())

# it seems that the stop_timezone column is not used in the feed since all values are NaN, so we will drop the column from the dataframe
stops = stops.drop(columns=["stop_timezone"])

# wheelchair_boarding
print("Distribution of wheelchair_boarding column in stops dataframe:")
print(stops["wheelchair_boarding"].value_counts())
print("-" * 50)

# so it looks like most stops support wheelchair boarding (value 1), but some do not (value 2), and there are a few that have unknown wheelchair boarding status (value 0) 

# level_id
print("Distribution of level_id column in stops dataframe:")
print(stops["level_id"].value_counts())
print("-" * 50)

# it seems that the level_id column is not used in the feed since all values are NaN, so we will drop the column from the dataframe
stops = stops.drop(columns=["level_id"])

print("Updated Data types of stops dataframe:")
print(stops.dtypes)
print("-" * 50)

Range of stop_timezone column in stops dataframe:
nan nan
Distribution of wheelchair_boarding column in stops dataframe:
wheelchair_boarding
1    8020
2    1396
0       7
Name: count, dtype: int64
--------------------------------------------------
Distribution of level_id column in stops dataframe:
Series([], Name: count, dtype: int64)
--------------------------------------------------
Updated Data types of stops dataframe:
stop_id                  int64
stop_code                int64
stop_name               object
stop_lat               float64
stop_lon               float64
location_type            int64
wheelchair_boarding      int64
dtype: object
--------------------------------------------------


In [131]:
# lets look at the presence of null values and empty strings in the rows of the stop 

print("Number of null values in each column of stops dataframe:")
print(stops.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of stops dataframe:")
object_columns = stops.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (stops[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# overall, looks like the feed for stops has complete data

Number of null values in each column of stops dataframe:
stop_id                0
stop_code              0
stop_name              0
stop_lat               0
stop_lon               0
location_type          0
wheelchair_boarding    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of stops dataframe:
stop_name: 0
--------------------------------------------------


#### Routes.txt

We will now perform the same cleaning process to the routes table

In [132]:
# print data types of routes dataframe

print("Data types of routes dataframe:")
print(routes.dtypes)
print("-" * 50)

Data types of routes dataframe:
route_id              int64
agency_id             int64
route_short_name      int64
route_long_name      object
route_desc          float64
route_type            int64
route_url           float64
route_color          object
route_text_color     object
dtype: object
--------------------------------------------------


In [133]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in routes.columns:
    unique_values = routes[col].unique()
    print(f"Unique values in {col} column of routes dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in route_id column of routes dataframe:
[ 10 100 101 102 103 104 105 106 107 108 109  11 110 111 112 113 114 115
 116 117 118 119  12 120 121 122 123 124 125 126 127 129  13 130 131 132
 133 134 135  14 149  15 151 154 156 158  16 160 161 162 164 165 166 167
 168 169  17 171  18 184 185 189  19 191  20 203  21  22  23  24  25  26
  27  28  29  30 300 301 304 305 306 307  31 310 312 315 316  32 320 322
 324 325 329  33 334 335 336 337 339  34 340 341 343  35 352 353 354  36
 363 365  37  38 384 385 386  39 395 396  40  41  42  43  44  45  46  47
  48  49  50 501 503 504 505 506 507 508 509  51 510 511 512  52  53  54
  55  57  59  60  61  62  63  64  65  66  67  68  69   7  70  71  72  73
  74  75  76  77  78  79   8  80 805  82  83 830  84  85  86  87  88  89
   9  90 900 902 903 904 905 906  91  92 924 925 927 929  93 935 937 938
 939  94 941 944 945  95 952 953 954  96 960 968  97  98 984 985 986 989
  99 995 996   1   2   4 400 402 403 404 405 406   5   6]
------------

In [134]:
# drop route_desc and route_url columns since they are not used in the feed
routes = routes.drop(columns=["route_desc", "route_url"])

print("Updated Data types of routes dataframe:")
print(routes.dtypes)
print("-" * 50)

Updated Data types of routes dataframe:
route_id             int64
agency_id            int64
route_short_name     int64
route_long_name     object
route_type           int64
route_color         object
route_text_color    object
dtype: object
--------------------------------------------------


In [146]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

for col in routes.columns:
    if col == "route_id":
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)
    elif routes[col].dtype == "int64":
        print(f"Range of {col} column in routes dataframe: {routes[col].min()} to {routes[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)
    elif routes[col].dtype == "float64":
        print(f"Range of {col} column in routes dataframe: {routes[col].min()} to {routes[col].max()}")
        print("-" * 50)
    elif routes[col].dtype == "object":
        print(f"Distribution of {col} column in routes dataframe:")
        print(routes[col].value_counts(dropna=False))
        print("-" * 50)

Distribution of route_id column in routes dataframe:
route_id
10     1
55     1
59     1
60     1
61     1
      ..
306    1
307    1
31     1
310    1
6      1
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Range of agency_id column in routes dataframe: 1 to 1
--------------------------------------------------
Distribution of agency_id column in routes dataframe:
agency_id
1    230
Name: count, dtype: int64
--------------------------------------------------
Range of route_short_name column in routes dataframe: 1 to 996
--------------------------------------------------
Distribution of route_short_name column in routes dataframe:
route_short_name
10     1
55     1
59     1
60     1
61     1
      ..
306    1
307    1
31     1
310    1
6      1
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Distribution of route_long_name column in routes dataframe:
route_long_name
Bathurst            3
Lawrence East  

In [136]:
# There are some obervations we can make from this

# 1) There are 230 unique rows in the routes dataframe; each row seems to correspnd to a unique value of route_id (230 unique route_ids). This is a good sign
#    that the primary key is unique

# 2) agency_id contains only 1 unique value (1) meaning that the only transit agency managing routes in this feed is the TTC

# 3) it seems each route has a unique route_short_name (equal in value to route_id) but some routes share the same route_long_name
#    This makes sense since the same bus routes, for example, can have different route numbers but end up at the same destination
#.   For example route 307 and 511 are different routes but both end up at Bathurst station, so they share the same route_long_name

# 4) There are 3 unique route_types in the TTC feed; 3 (bus), 1 (subway), and 4 (streetcar) which makes sense since the TTC operates these 3 modes of transportation
#    There are more bus routes than subway and streetcar routes which also makes sense since the bus network is usually more extensive than subway and streetcar networks in a city
#.   The 3 subway routes line up with the 3 subway lines in Toronto (Yonge-University-Spadina, Bloor-Danforth, and Sheppard) which is another good sign that the feed is accurate and complete


In [137]:
# looking at routes with route_long_name = "Bathurst"
print("Routes with route_long_name 'Bathurst':")
print(routes[routes["route_long_name"] == "Bathurst"])
print("-" * 50)

# looking at routes with route_type = 1 (subway)
print("Routes with route_type 1 (subway):")
print(routes[routes["route_type"] == 1])
print("-" * 50)


Routes with route_long_name 'Bathurst':
     route_id  agency_id  route_short_name route_long_name  route_type  \
81        307          1               307        Bathurst           3   
139       511          1               511        Bathurst           0   
157         7          1                 7        Bathurst           3   

    route_color route_text_color  
81       0054A6           FFFFFF  
139      ED1C24           FFFFFF  
157      ED1C24           FFFFFF  
--------------------------------------------------
Routes with route_type 1 (subway):
     route_id  agency_id  route_short_name        route_long_name  route_type  \
219         1          1                 1  Yonge-University Line           1   
220         2          1                 2    Bloor-Danforth Line           1   
221         4          1                 4          Sheppard Line           1   

    route_color route_text_color  
219      D5C82B           000000  
220      008000           FFFFFF  
221    

In [138]:
# lets look at the presence of null values and empty strings in the rows of the routes dataframe

print("Number of null values in each column of routes dataframe:")
print(routes.isnull().sum())
print("-" * 50)

# lets look at the presence of null values and empty strings in the rows of the routes dataframe
print("Number of empty strings in object dtype columns of routes dataframe:")
object_columns = routes.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (routes[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

# overall, looks like the feed for routes has complete data

Number of null values in each column of routes dataframe:
route_id            0
agency_id           0
route_short_name    0
route_long_name     0
route_type          0
route_color         0
route_text_color    0
dtype: int64
--------------------------------------------------
Number of empty strings in object dtype columns of routes dataframe:
route_long_name: 0
route_color: 0
route_text_color: 0
--------------------------------------------------


#### Trips.txt

In [140]:
# print data types of trips dataframe

print("Data types of trips dataframe:")
print(trips.dtypes)
print("-" * 50)

Data types of trips dataframe:
trip_id                   int64
route_id                  int64
service_id                int64
trip_headsign            object
trip_short_name          object
direction_id              int64
block_id                  int64
shape_id                 object
wheelchair_accessible     int64
dtype: object
--------------------------------------------------


In [141]:
# lets first identify columns that are not used in the feed and drop them from the dataframe to simplify our analysis
# we can identify these columns by looking at the unique values in each column and seeing if they are all NaN or empty strings

for col in trips.columns:
    unique_values = trips[col].unique()
    print(f"Unique values in {col} column of trips dataframe:")
    print(unique_values)
    print("-" * 50)

Unique values in trip_id column of trips dataframe:
[106986020 118048020 102340020 ... 133251418 133251419 133251420]
--------------------------------------------------
Unique values in route_id column of trips dataframe:
[ 10 100 101 102 103 104 105 106 107 108 109  11 110 111 112 113 114 115
 116 117 118 119  12 120 121 122 123 124 125 126 127 129  13 130 131 132
 133 134 135  14 149  15 151 154 156 158  16 160 161 162 164 165 166 167
 168 169  17 171  18 184 185 189  19 191  20 203  21  22  23  24  25  26
  27  28  29  30 300 301 304 305 306 307  31 310 312 315 316  32 320 322
 324 325 329  33 334 335 336 337 339  34 340 341 343  35 352 353 354  36
 363 365  37  38 384 385 386  39 395 396  40  41  42  43  44  45  46  47
  48  49  50 501 503 504 505 506 507 508 509  51 510 511 512  52  53  54
  55  57  59  60  61  62  63  64  65  66  67  68  69   7  70  71  72  73
  74  75  76  77  78  79   8  80 805  82  83 830  84  85  86  87  88  89
   9  90 900 902 903 904 905 906  91  92 924 925

In [147]:
# each dytype seems to be appropriate for the type of data in each column
# lets now look at the distribution of values for each column as well

# for the primary key, we will look at the number of unique values and their frequencies
# for integer columns we will look at the range and possibly mean
# for float values we will look at the range (float values usually will represent geographical coordinates so taking the mean/median doesnt make sense)
# for object/string values we will look at the unique values and their frequencies/mode

for col in trips.columns:
    if col == "trip_id":
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)
    elif trips[col].dtype == "int64":
        print(f"Range of {col} column in trips dataframe: {trips[col].min()} to {trips[col].max()}")
        print("-" * 50)
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)
    elif trips[col].dtype == "float64":
        print(f"Range of {col} column in trips dataframe: {trips[col].min()} to {trips[col].max()}")
        print("-" * 50)
    elif trips[col].dtype == "object":
        print(f"Distribution of {col} column in trips dataframe:")
        print(trips[col].value_counts(dropna=False))
        print("-" * 50)


Distribution of trip_id column in trips dataframe:
trip_id
106986020    1
80251020     1
11708020     1
126289070    1
117554020    1
            ..
18410080     1
86115070     1
39406010     1
10436080     1
133251420    1
Name: count, Length: 135412, dtype: int64
--------------------------------------------------
Range of route_id column in trips dataframe: 1 to 996
--------------------------------------------------
Distribution of route_id column in trips dataframe:
route_id
504    3246
63     2533
47     2393
1      2276
2      2206
       ... 
406      16
405      14
400      14
403      12
830       2
Name: count, Length: 230, dtype: int64
--------------------------------------------------
Range of service_id column in trips dataframe: 1 to 7001
--------------------------------------------------
Distribution of service_id column in trips dataframe:
service_id
1       39810
2       33519
3       30298
5       29810
4        1929
4501       16
7001       16
4401       14
Name: coun

In [ ]:
# There are some obervations we can make from this

# 1) There are 135412 unique rows in the trips dataframe; each row seems to correspnd to a unique value of trip_id (135412 unique trip_ids)
#    This is a good sign that the primary key is unique

# 2) The route_id column contains 230 unique values ranging from 1 to 996. This matches the number of unique route_ids in the routes
#    dataframe and the range of route_id's in the routes dataframe. This means all routes will have one or more trips scheduled, which is a good
#    sign that the feed is complete

# 3) it seems that all scheduled trips will follow 8 possible weekly schedules (service_id's). 

# 4) it seems many trips share the same trip_headsign (ranging from 1 to thousands sharing the same headsign), this makes sense since
#    many trips from different routes, scheduled at different times, can share the same destination and therefore the same headsign

# 5) Most trips do not have a specified trip_short_name (value NaN) but the rest seem to have an arbritary letter assigned as a trip_short_name
#    There doesnt seem to be much documentation of what exactly the trip_short_name represents in the TTC feed, but they seem to refer to the
#.   letter associated with a bus trip in the TTC system i.e. the 102 "D" (see below)

# 6) The range of direction_id is from 0 to 1 which makes sense since direction_id entails the direction of travel of the trip 
#    (0 vs 1 meaning eastbound vs westbound, etc.)

# 7) block_id represents the block of sequential trips made by the same vehicle, so it makes sense that there are fewer unique blocks than trips
#    it is suprising that hundreds of trips share the same block_id, but this could just be looped routes or the same sequence of routes being 
#    travelled through by different veichles at different schedules

# 8) 

In [152]:
# what is the range of route_id values in the routes dataframe
print("Range of route_id values in routes dataframe:")
print(routes["route_id"].min(), "to", routes["route_id"].max())
print("-" * 50)

# print out records with a trip_short_name of "D" 
print("Trips with trip_short_name 'D':")
print(trips[trips["trip_short_name"] == "D"].head(10).to_string())
print("-" * 50)

Range of route_id values in routes dataframe:
1 to 996
--------------------------------------------------
Trips with trip_short_name 'D':
       trip_id  route_id  service_id                                    trip_headsign trip_short_name  direction_id  block_id    shape_id  wheelchair_accessible
2005  74998070       102           2  North - 102D Markham Rd towards Major Mackenzie               D             1  10222220  shp-102-08                      1
2006  23309020       102           1  North - 102D Markham Rd towards Major Mackenzie               D             1   1020990  shp-102-08                      1
2007  71179080       102           5  North - 102D Markham Rd towards Major Mackenzie               D             1  10214140  shp-102-08                      1
2008  23577070       102           2  North - 102D Markham Rd towards Major Mackenzie               D             1  10214140  shp-102-08                      1
2009  75263020       102           1  North - 102D Markha

In [145]:
# lets look at the presence of null values and empty strings in the rows of the trips dataframe

print("Number of null values in each column of trips dataframe:")
print(trips.isnull().sum())
print("-" * 50)

print("Number of empty strings in object dtype columns of trips dataframe:")
object_columns = trips.select_dtypes(include=["object"]).columns
for col in object_columns:
    num_empty_strings = (trips[col] == "").sum()
    print(f"{col}: {num_empty_strings}")
print("-" * 50)

Number of null values in each column of trips dataframe:
trip_id                      0
route_id                     0
service_id                   0
trip_headsign                0
trip_short_name          92831
direction_id                 0
block_id                     0
shape_id                     0
wheelchair_accessible        0
dtype: int64
--------------------------------------------------


In [139]:
print("Data types of stop_times dataframe:")
print(stop_times.dtypes)
print("-" * 50)

print("Data types of shapes dataframe:")
print(shapes.dtypes)
print("-" * 50)

print("Data types of calendar dataframe:")
print(calendar.dtypes)
print("-" * 50)

print("Data types of agency dataframe:")
print(agency.dtypes)
print("-" * 50)

print("Data types of calendar_dates dataframe:")
print(calendar_dates.dtypes)
print("-" * 50)

print("Data types of route_types dataframe:")
print(route_types.dtypes)
print("-" * 50)

print("Data types of feed_info dataframe:")
print(feed_info.dtypes)
print("-" * 50)

Data types of stop_times dataframe:
trip_id                  int64
arrival_time            object
departure_time          object
stop_id                  int64
stop_sequence            int64
stop_headsign           object
pickup_type              int64
drop_off_type            int64
shape_dist_traveled    float64
timepoint              float64
dtype: object
--------------------------------------------------
Data types of shapes dataframe:
shape_id                object
shape_pt_lat           float64
shape_pt_lon           float64
shape_pt_sequence        int64
shape_dist_traveled    float64
dtype: object
--------------------------------------------------
Data types of calendar dataframe:
service_id    int64
monday        int64
tuesday       int64
wednesday     int64
thursday      int64
friday        int64
saturday      int64
sunday        int64
start_date    int64
end_date      int64
dtype: object
--------------------------------------------------
Data types of agency dataframe:
agency